# Figure: Processed Image Histogram Comparison

Compares the input processed image with the re-processed prediction to validate
reconstruction quality. If the inversion is good, `pred ≈ imgs`.

**Rows:** input image | prediction  
**Columns:** image | pixel histogram

**Requires:** `nb_run.ipynb`

Output: `build/processed_hist_cmp.png`

In [ ]:
%%capture
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    DRIVE_ROOT = "/content/drive/MyDrive/thesis"  # <-- edit if your Drive path differs

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", f"{DRIVE_ROOT}/notebooks/requirements.txt",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "-e", DRIVE_ROOT,
    ])

    # Change to figures/ so load_results() can find the cached data
    os.chdir(f"{DRIVE_ROOT}/figures")


In [ ]:
import sys, os, glob, joblib

import numpy as np
import matplotlib.pyplot as plt

def load_results(results_dir="results"):
    hits = sorted(glob.glob(f"{results_dir}/*/imgs.joblib"))
    if not hits:
        raise FileNotFoundError(f"No results in '{results_dir}'. Run nb_run.ipynb first.")
    d = os.path.dirname(hits[-1])
    print(f"Loading from {d}")
    return {
        "imgs": joblib.load(f"{d}/imgs.joblib"),
        "pred": joblib.load(f"{d}/pred.joblib"),
    }

In [ ]:
r = load_results()
imgs, pred = r["imgs"], r["pred"]

IDX = 0  # which sample to show

In [ ]:
img_i  = np.asarray(imgs[IDX])
pred_i = np.asarray(pred[IDX])

fig, ax = plt.subplots(2, 2, figsize=(14, 12), sharey="col")

ax[0, 0].set_title("Input processed image")
ax[0, 0].imshow(img_i, cmap="gray", vmin=0, vmax=1)
ax[0, 0].axis("off")
ax[0, 1].hist(img_i.flatten(), bins=70, color="steelblue", alpha=0.8)
ax[0, 1].set_xlabel("Pixel value")
ax[0, 1].set_ylabel("Frequency")
ax[0, 1].set_xlim(0, 1)

ax[1, 0].set_title("Prediction (forward of recovered TXM)")
ax[1, 0].imshow(pred_i, cmap="gray", vmin=0, vmax=1)
ax[1, 0].axis("off")
ax[1, 1].hist(pred_i.flatten(), bins=70, color="orange", alpha=0.8)
ax[1, 1].set_xlabel("Pixel value")
ax[1, 1].set_ylabel("Frequency")
ax[1, 1].set_xlim(0, 1)

fig.tight_layout()
fig.savefig("build/processed_hist_cmp.png", bbox_inches="tight")
plt.show()

In [ ]:
# Overlay histogram for direct comparison across all samples
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.asarray(imgs).flatten(), bins=100, alpha=0.5, label="Input",      color="steelblue", density=True)
ax.hist(np.asarray(pred).flatten(), bins=100, alpha=0.5, label="Prediction", color="orange",    density=True)
ax.set_xlabel("Pixel value")
ax.set_ylabel("Density")
ax.set_title(f"Population comparison ({imgs.shape[0]} images)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("build/processed_hist_cmp_population.png", bbox_inches="tight")
plt.show()